# Final session (v9): YOLO11s baseline with the adopted YOLO11m configuration, then ONE test evaluation per model
YOLO11m weights come from run 2 (notebook v8) via the dataset prakyats/wood-yolo11m-weights. Test split is touched only here.

In [ ]:
!pip -q install -U ultralytics
!nvidia-smi --query-gpu=name,memory.total --format=csv
import ultralytics, torch, os, glob, shutil, time
print(ultralytics.__version__, torch.__version__, torch.cuda.device_count(), 'GPUs')
print('inputs:', os.listdir('/kaggle/input'))
SRC = '/kaggle/input/wood-defects-clean/dataset_clean'
assert os.path.isdir(SRC + '/train/images'), os.listdir('/kaggle/input')
hits = glob.glob('/kaggle/input/**/train_yolo11m.py', recursive=True)
assert hits, 'train_yolo11m.py not found under /kaggle/input - is wood-defects-scripts attached?'
SCRIPTS = os.path.dirname(hits[0]); print('scripts dir:', SCRIPTS, os.listdir(SCRIPTS))
# stage on local disk (/kaggle/tmp is not persisted): mounted input reads at ~36 MB/s and is read-only (no label cache)
DATA = '/kaggle/tmp/dataset_clean'
if not os.path.isdir(DATA + '/train/images'):
    t = time.time(); shutil.copytree(SRC, DATA); print(f'copied dataset to local disk in {time.time()-t:.0f}s')
lines = open(DATA + '/data.yaml').read().splitlines()
lines = [('path: ' + DATA) if l.startswith('path:') else l for l in lines]
os.makedirs('/kaggle/working/cfg', exist_ok=True)
with open('/kaggle/working/cfg/data.yaml', 'w') as f:
    f.write('\n'.join(lines) + '\n')
print('\n'.join(lines))

In [ ]:
# restore a previous session's outputs if attached as input (Add Input -> this notebook's output)
import glob, shutil
for prev in glob.glob('/kaggle/input/*/outputs/yolo11m_1024rect'):
    if not os.path.exists('/kaggle/working/outputs/yolo11m_1024rect'):
        shutil.copytree(prev, '/kaggle/working/outputs/yolo11m_1024rect')
        print('restored', prev)

In [ ]:
# YOLO11s baseline, identical to the adopted YOLO11m config (run 2): batch 32, lr0 0.001, patience 40, 1024 rect, seed 42.
import subprocess, sys
RUN_S = 'yolo11s_1024rect_b32_lr001'
cmd = [sys.executable, f'{SCRIPTS}/train_yolo11m.py', '--data', '/kaggle/working/cfg/data.yaml',
       '--project', '/kaggle/working/outputs', '--name', RUN_S, '--device', '0', '--model', 'yolo11s.pt',
       '--batch', '32', '--lr0', '0.001', '--patience', '40', '--persist', '/kaggle/working/persist']
print(' '.join(cmd)); rc = subprocess.call(cmd); print('exit', rc)
assert os.path.exists(f'/kaggle/working/outputs/{RUN_S}/results.csv'), 'YOLO11s training produced no results.csv - see log above'
print(open(f'/kaggle/working/outputs/{RUN_S}/results.csv').read()[-800:])

## Test evaluation: exactly once per model. No decision may be made from these numbers.

In [ ]:
W_M = glob.glob('/kaggle/input/**/yolo11m_1024rect_b32_lr001_best.pt', recursive=True)
assert W_M, 'YOLO11m run-2 weights not found - is wood-yolo11m-weights attached?'
W_S = f'/kaggle/working/outputs/{RUN_S}/weights/best.pt'
assert os.path.exists(W_S)
for tag, w in [('yolo11m_run2', W_M[0]), ('yolo11s_baseline', W_S)]:
    out = f'/kaggle/working/eval/{tag}'
    rc = subprocess.call([sys.executable, f'{SCRIPTS}/evaluate.py', '--weights', w, '--data', '/kaggle/working/cfg/data.yaml',
                          '--split', 'test', '--device', '0', '--out', out])
    print(tag, 'exit', rc)
    assert os.path.exists(f'{out}/test_metrics.md'), f'{tag}: no test_metrics.md'
    print(open(f'{out}/test_metrics.md').read())